In [10]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from tools import save_report_to_md, web_search_tool

In [12]:
model_client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
)

In [13]:
research_planner = AssistantAgent(
    name="ResearchPlanner",
    model_client=model_client,
    description="A strategic research coordinator that breaks down complex questions into research subtasks.",
    system_message="""
    You are a research planning specialist. Your job is to create a focused research plan.
    
    For each research question, create a FOCUSED research plan with:
    
    1. **Core Topics**: 2-3 main areas to investigate
    2. **Search Queries**: Create 3-5 specific search queries covering:
        - Latest developments and news
        - Key statistics or data
        - Expert analysis or studies
        - Future outlook
    
    Keep the plan focused and achievable. Quality over quantity.
    """,
)

research_agent = AssistantAgent(
    name="ResearchAgent",
    model_client=model_client,
    tools=[web_search_tool, save_report_to_md],
    description="A web research specialist that searches and extracts content",
    system_message="""
    You are a web research specialist. Your job is to conduct focused searches based on the research plan.
    
    RESEARCH STRATEGY:
    1. **Execute 3-5 search** from the research plan
    2. **Extract Key information** from the search results:
        - Main facts and statistics
        - Recent developments
        - Expert opinions
        - Important context
        
    3. **Quality focus**:
        - Prioritize authoritative sources
        - Loog for recent information (within 2 years)
        - Note diverse perspectives
    
    After completing the searches from the plan, summarize what you found. Your goal is to gather 5-10 quality sources.
    """,
)

research_analyst = AssistantAgent(
    name="ResearchAnalyst",
    model_client=model_client,
    description="An expert analyst that creates reports",
    system_message="""
    You are a research analyst. Create a comprehensive report from the gathered research.
    
    CREATE A RESEARCH REPORT with;
    
    ## Executive Summary
    - Key findings and conclusions
    - Main insights
    
    ## Background & Current State
    - Current landscape
    - Recent developments
    - Key statistics and data
    
    ## Analysis & Insights
    - Main trends
    - Different perspectives
    - Expert opinions
    
    ## Future Outlook
    - Emerging trends
    - Predictions
    - Implications
    
    ## Sources
    - List all sources used
    
    Write a clear, well-structured report based on the research gathered. End with "REPORT_COMPLETE" when finished.
    """,
)

quality_reviewer = AssistantAgent(
    name="QualityReviewer",
    tools=[save_report_to_md],
    model_client=model_client,
    description="A quality assurance specialist that evaluates research completeness and accuracy",
    system_message="""
    Your are a quality reviewer. Your job is to check if the research analyst has produced a complete research report.
    
    Look for:
    - A comprehensive research report from the research analyst that ends with "REPORT_COMPLETE"
    - The research question is fully answered
    - Sources are cited and reliable
    - The report includes summary, key information, analysis, and sources
    
    When you see a complete research report that ends with "REPORT_COMPLETE":
    1. Firs, use the save_report_to_md tool to save the report to report.md
    2. Then say: "The research is complete. The report has been saved to report.md. Please review the report and let me know if you approve it or need additional research."
    
    If the research analyst has NOT yet created a complete report, tell them to create one now.
    """,
)

research_enhancer = AssistantAgent(
    name="ResearchEnhancer",
    model_client=model_client,
    description="A specialist that identifies critical gaps only",
    system_message="""
    You are a research enhancement specialist. Your job is to identify ONLY CRITICAL gaps.
    
    Review the research and ONLY suggest additional searches if there are MAJOR gaps like:
    - Completely missing recent developments
    - No statistics or data at all
    - Missing a crucial perspective that was specifically asked for
    
    If the research covers the basics reasonably well, say: "The research is sufficient to proceed with the report."
    
    Only suggest 1-2 additional if absolutely necessary. We prioritize getting a good report done rather than perfect coverage.
    """,
)

user_proxy = UserProxyAgent(
    name="UserProxy",
    description="Human reviewer who can request additional research or approve final results",
    input_func=input,
)

In [14]:
# selector_prompt 의  {roles} 와 {history} 부분은 AutoGen 에 의해서 자동으로 교체된다. 예약어다.
# AutoGen이 모든 participant를 가져와서 {roles} 플레이스 홀더에 넣어준다.
# AutoGen이 {history}를 보게 되면 모든 대화 히스토리를 해당 플레이스 홀더에 넣어준다.

selector_prompt = """
Choose the best agent for the current task based on the conversation history:

{roles}

Current conversation:
{history}

Available agents:
- research_planner: Plan the research approach (ONLY at the start)
- research_agent: Search for and extract content from web sources (after planning)
- research_enhancer: Identify CRITICAL gaps only (use sparingly)
- research_analyst: Write the final research report
- quality_reviewer: Check if a complete report exists
- user_proxy: Ask the human for feedback

WORKFLOW:
1. If no planning done yet → select research_planner
2. If planning done but no research → select research_agent  
3. After research_agent completes initial searches → select research_enhancer ONCE
4. If enhancer says "sufficient to proceed" → select research_analyst
5. If enhancer suggests critical searches → select research_agent ONCE more then research_analyst
6. If research_analyst said "REPORT_COMPLETE" → select quality_reviewer
7. If quality_reviewer asked for user feedback → select user_proxy

IMPORTANT: After research_agent has searched 2 times maximum, proceed to research_analyst regardless.

Pick the agent that should work next based on this workflow."""

In [15]:
text_termination = TextMentionTermination(text="APPROVED")
max_message_termination = MaxMessageTermination(max_messages=20)
termination_condition = text_termination | max_message_termination

team = SelectorGroupChat(
    participants=[
        research_agent,
        research_analyst,
        research_enhancer,
        research_planner,
        quality_reviewer,
        user_proxy,
    ],
    selector_prompt=selector_prompt,
    model_client=model_client,
    termination_condition=termination_condition,
    # allow_repeated_speaker=True, # 한 에이전트가 여러번 말하고 반복할 수 있다는 뜻임.
)

In [16]:
await Console(
    team.run_stream(
        task="Research about the new development in Nuclear Energy",
    )
)

---------- TextMessage (user) ----------
Research about the new development in Nuclear Energy
---------- TextMessage (ResearchPlanner) ----------
Below are five focused research questions that cover the most important “new developments” in nuclear energy. For each question I give 2–3 core topics to investigate and 3–5 ready-to-run search queries that cover: latest news, key statistics/data, expert analysis/studies, and future outlook. Use these queries in Google, Google Scholar, or your institution’s library; add date filters (e.g., 2023–2025) to prioritize the newest material.

1) Research question: What are the latest developments in Small Modular Reactors (SMRs) and advanced fission reactor designs?
- Core Topics:
  - SMR designs, deployment status, and supply-chain readiness (NuScale, Rolls‑Royce, Korea, China)
  - Licensing, regulatory approvals, and first-of-a-kind (FOAK) projects
  - Cost estimates, manufacturing scale-up, and market prospects
- Search queries:
  - Latest develo

TaskResult(messages=[TextMessage(id='53606c80-aaca-4ced-b491-163f2ce44df5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 10, 9, 6, 15, 57, 933099, tzinfo=datetime.timezone.utc), content='Research about the new development in Nuclear Energy', type='TextMessage'), TextMessage(id='ff929c89-02e0-475c-9fd6-2107cd905b93', source='ResearchPlanner', models_usage=RequestUsage(prompt_tokens=129, completion_tokens=2201), metadata={}, created_at=datetime.datetime(2025, 10, 9, 6, 16, 48, 340068, tzinfo=datetime.timezone.utc), content='Below are five focused research questions that cover the most important “new developments” in nuclear energy. For each question I give 2–3 core topics to investigate and 3–5 ready-to-run search queries that cover: latest news, key statistics/data, expert analysis/studies, and future outlook. Use these queries in Google, Google Scholar, or your institution’s library; add date filters (e.g., 2023–2025) to prioritize the newest materi